In [ ]:
#LSTM+Word2vec
!pip install -q gensim tensorflow scikit-learn pandas matplotlib seaborn

import os
import re
import random
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
    roc_auc_score
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Update paths as needed
DATA_PATH = "AmHQA.csv"
WORD2VEC_PATH = "amharic_word2vec.bin"
df = pd.read_csv(DATA_PATH, encoding="utf-8")
question_col = next(
    (c for c in ["question", "Question"] if c in df.columns), None)
label_col = next(
    (c for c in ["category", "Category", "label", "Label"]
     if c in df.columns), None
)

if question_col is None or label_col is None:
    raise ValueError(f"Unexpected dataset columns: {df.columns.tolist()}")

df = df[[question_col, label_col]].copy()
df.columns = ["question", "category"]
df = df.dropna()

df["question"] = df["question"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()
df = df[
    (df["question"] != "") & (df["category"] != "")
].drop_duplicates().reset_index(drop=True)

print("Dataset size:", df.shape)
print(df["category"].value_counts())
#Load pretrained Word2Vec embeddings
word_vectors = KeyedVectors.load_word2vec_format(
    WORD2VEC_PATH,
    binary=True
)

print("Word2Vec vocabulary size:", len(word_vectors))
print("Embedding dimension:", word_vectors.vector_size)
#Tokenize and split the data
MAX_LEN = 40
MAX_WORDS = 50000

def clean_text(text):
    return re.sub(r"\s+", " ", str(text).strip())

df["question"] = df["question"].apply(clean_text)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["category"])
num_classes = len(label_encoder.classes_)

# 80% train, 10% validation, 10% test
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df["question"].to_numpy(),
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    filters=""
)
tokenizer.fit_on_texts(X_train_text)

def encode_questions(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

X_train = encode_questions(X_train_text)
X_val = encode_questions(X_val_text)
X_test = encode_questions(X_test_text)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
embedding_dim = word_vectors.vector_size

# Construct embedding matrix from pretrained vectors
embedding_matrix = np.random.normal(
    0, 0.05, (vocab_size, embedding_dim)
).astype(np.float32)

embedding_matrix[0] = 0.0
matched_words = 0

for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    if word in word_vectors:
        embedding_matrix[idx] = word_vectors[word]
        matched_words += 1

print("\nClasses:", list(label_encoder.classes_))
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Matched Word2Vec words:", matched_words)
print(
    "Embedding coverage:",
    f"{100 * matched_words / max(vocab_size - 2, 1):.2f}%"
)
#Build the LSTM classifier
inputs = Input(shape=(MAX_LEN,), name="question_input")

x = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=False,
    name="word2vec_embedding"
)(inputs)

x = LSTM(
    128,
    dropout=0.2,
    recurrent_dropout=0.0,
    name="lstm"
)(x)

x = Dropout(0.4)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)

outputs = Dense(
    num_classes,
    activation="softmax",
    name="class_probabilities"
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
#Train for 10 epochs
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
))
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(num_classes)
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.title("LSTM + Word2Vec Confusion Matrix")
plt.tight_layout()
plt.show()
y_test_bin = label_binarize(
    y_test,
    classes=np.arange(num_classes)
)

plt.figure(figsize=(9, 7))
class_aucs = []

for i, class_name in enumerate(label_encoder.classes_):
    # A class needs positive and negative test examples
    if len(np.unique(y_test_bin[:, i])) < 2:
        print(f"Skipping ROC for {class_name}: only one binary outcome.")
        continue

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    class_auc = auc(fpr, tpr)
    class_aucs.append((i, class_auc))

    plt.plot(
        fpr, tpr, linewidth=2,
        label=f"{class_name} (AUC = {class_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("LSTM + Word2Vec ROC Curves")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Compute overall AUC when all classes are represented
if num_classes == 2:
    overall_auc = roc_auc_score(y_test, y_prob[:, 1])
    print(f"Binary ROC-AUC: {overall_auc:.4f}")
elif len(np.unique(y_test)) == num_classes:
    print(f"Macro ROC-AUC: {roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro'):.4f}")
    print(f"Weighted ROC-AUC: {roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted'):.4f}")
else:
    print("Overall multiclass AUC omitted because at least one class is missing from the test set.")

print("\nPer-class AUC:")
for i, score in class_aucs:
    print(f"{label_encoder.classes_[i]}: {score:.4f}")
epochs = range(1, 11)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs, history.history["accuracy"], marker="o", label="Training")
axes[0].plot(epochs, history.history["val_accuracy"], marker="s", label="Validation")
axes[0].set_title("Accuracy Across 10 Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_xticks(list(epochs))
axes[0].legend()

axes[1].plot(epochs, history.history["loss"], marker="o", label="Training")
axes[1].plot(epochs, history.history["val_loss"], marker="s", label="Validation")
axes[1].set_title("Loss Across 10 Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_xticks(list(epochs))
axes[1].legend()

plt.tight_layout()
plt.show()
model.save("LSTM_Word2Vec_Amharic_Classifier.keras")

with open("LSTM_Word2Vec_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("LSTM_Word2Vec_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Model and preprocessing objects saved.")